**Data Extraction & Transformation (Python)**

**Step 1: Source data loaded from CSV**

In [ ]:
import pandas as pd

# The original Google Drive link points to a viewer page, not the raw data.
# We need to convert it to a direct download link.
file_id = '19QzAIMQi3i4ggwWHXx325Y7aVIH3dyUx'
direct_download_url = f'https://drive.google.com/uc?export=download&id={file_id}'

df = pd.read_csv(direct_download_url)
df

**Data cleaning using Pandas---------------1.Duplicate Data removal**

In [ ]:
#Total number of rows
total_rows = len(df)
print(total_rows)

# Count total duplicate rows
num_duplicates = df.duplicated().sum()
print(num_duplicates)

In [ ]:
# View the duplicate rows
duplicate_data = df[df.duplicated(keep=False)]
duplicate_data


In [ ]:
# View the duplicate rows sort by duplicated rows
duplicate_data = df[df.duplicated(keep=False)].sort_values(by=list(df.columns))
duplicate_data

In [5]:
# Standard removal (returns a new DataFrame)
df_cleaned = df.drop_duplicates()

# Modify the existing DataFrame directly
df.drop_duplicates(inplace=True)
#Total number of rows after duplicates removal
total_rows = len(df)
print(total_rows)

23158


**Data cleaning using Pandas---------------2.Missing value handling**

In [ ]:
# Returns a Series with the count of nulls for every column
df.isnull().sum()


**Data cleaning using Pandas---------------3.Rating Normalization**

In [ ]:
# find out the unique data
df['rate'].unique()

In [ ]:
# removing the data from /
df['rate'] = df['rate'].str.split('/').str[0]
df

In [ ]:
# removing the space from the column
df['rate'] = df['rate'].str.strip()
df

In [ ]:
#turning errors like "NEW" or "-" into NaN
df['rate'] = pd.to_numeric(df['rate'], errors='coerce')
df

In [ ]:
# find out the unique data
df['rate'].unique()

In [ ]:
# Convert the cleaned rating to a decimal number
df['rate'] = df['rate'].astype(float)
df

In [ ]:
df.dtypes

In [14]:
from numpy import average
#Find Average Rating
average_rating = df['rate'].mean()
print(average_rating)
#

3.904002259690596


**Data cleaning using Pandas---------------4.Cost standardization**

In [ ]:
# To modify the existing DataFrame directly (in-place)
df.rename(columns={'approx_cost(for two people)': 'cost'}, inplace=True)
df

In [ ]:
#1. removing the comma to handle the numeric values
df['cost'] = df['cost'].str.replace(',', '')

# 2. Convert to numeric (coerce handles any weird strings by making them NaN)
df['cost'] = pd.to_numeric(df['cost'], errors='coerce')
df['cost']

**Data cleaning using Pandas---------------5.Feature engineering for pricing segments and rating categories**

In [ ]:
df

Data cleaning using Pandas---------------6.Cleaning

In [18]:
filtered_df = df[df['phone'].str.contains(r'\r|\n', na=False)]
filtered_df

,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,cost,listed_in(type),listed_in(city)
0,Jalsa,Yes,Yes,4.1,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,Buffet,Banashankari
4,Grand Village,No,No,3.8,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,Buffet,Banashankari
5,Timepass Dinner,Yes,No,3.8,286,+91 9980040002\r\n+91 9980063005,Basavanagudi,Casual Dining,"Onion Rings, Pasta, Kadhai Paneer, Salads, Sal...",North Indian,600,Buffet,Banashankari
6,Onesta,Yes,Yes,4.6,2556,080 48653961\r\n080 48655715,Banashankari,"Casual Dining, Cafe","Farmhouse Pizza, Chocolate Banana, Virgin Moji...","Pizza, Cafe, Italian",600,Cafes,Banashankari
7,Penthouse Cafe,Yes,No,4.0,324,+91 8884135549\r\n+91 9449449316,Banashankari,Cafe,"Pizza, Mocktails, Coffee, Nachos, Salad, Pasta...","Cafe, Italian, Continental",700,Cafes,Banashankari
...,...,...,...,...,...,...,...,...,...,...,...,...,...
23183,BAR BAR,No,Yes,4.1,1003,+91 7338630404\n+91 8067266651,Whitefield,"Bar, Casual Dining","Mocktails, Cocktails, Pizza, Chicken Tikka, Na...","Continental, North Indian, Italian",1500,Pubs and bars,Whitefield
23186,Oliver's Pub & Diner,Yes,Yes,3.9,548,+91 8043691111\n+91 8028026519,Whitefield,"Pub, Casual Dining","Pizza, Beer, Cocktails, Nachos, Pasta, Moo Bur...","Finger Food, American, Continental, Burger, Pizza",1500,Pubs and bars,Whitefield
23187,Smaaash,No,Yes,4.0,189,+91 9900069474\n+91 7899644468,Whitefield,"Casual Dining, Pub","Pizza, Beer","North Indian, Pizza, Chinese",1500,Pubs and bars,Whitefield
23188,Izakaya Gastro Pub,Yes,Yes,3.8,128,+91 7625087121\n+91 8050587483,Whitefield,"Bar, Casual Dining","Beer, Chicken Guntur, Paneer Tikka, Fish, Nood...","North Indian, Continental, Mediterranean",1200,Pubs and bars,Whitefield


In [ ]:
# Regex: find \r that is NOT followed by \n
lone_r_mask = df['phone'].str.contains(r'\r(?!\n)', regex=True, na=False)
df_lone_r = df[lone_r_mask]
df_lone_r

In [20]:
idx = df.columns.get_loc('phone')
print("Phone index is " + str(idx))

Phone index is 5


In [ ]:
#phone number cleaning
#1. Add a alternative phone column after phone
df.insert(6, 'primary_phone', "")
df
#df['phone'] = df['phone'].str.split('\r\n').str[0]
#df

In [ ]:
#phone number cleaning
#1. Add a alternative phone column after phone
df.insert(7, 'secondary_phone', "")
df
#df['phone'] = df['phone'].str.split('\r\n').str[0]
#df

In [ ]:
#1. search for the \r\n in the phone column data and split the data and store to 2 different columns
df[['primary_phone', 'secondary_phone']] = df['phone'].str.split('\r\n', expand=True)
df

In [ ]:
#2. search for the \n in the phone column data and split the data and store to 2 different columns
df[['primary_phone', 'secondary_phone']] = df['phone'].str.split('\n', expand=True)
df

In [ ]:
#1. removing the \r and \n to clean the phone number
df['primary_phone'] = df['primary_phone'].str.replace('\r', '')
df['primary_phone'] = df['primary_phone'].str.replace('\n', '')
df['secondary_phone'] = df['secondary_phone'].str.replace('\r', '')
df['secondary_phone'] = df['secondary_phone'].str.replace('\n', '')
df


In [ ]:
null_counts = df.isnull().sum()
print(null_counts)

In [ ]:
filtered_df = df[df['location'].str.contains(r',', na=False)]
filtered_df

In [ ]:
# import sqlite3

# # 1. Connect to a database (creates it if it doesn't exist)
# conn = sqlite3.connect('UberEats.db')

# # 2. Create a cursor object to execute commands
# cursor = conn.cursor()

# # This creates the table 'new_table' automatically from your df
# df.to_sql('restaurant', conn, if_exists='replace', index=False)

# # 6. Fetch and display data
# cursor.execute('SELECT * FROM restaurant')
# rows = cursor.fetchall()
# for row in rows:
#     print(row)

# # 7. Close the connection when finished
# conn.close()

In [33]:
# import sqlite3
# import pandas as pd

# with sqlite3.connect('UberEats.db') as conn:
#     # This single line handles the query AND the formatting
#     dfsql = pd.read_sql_query("SELECT * FROM restaurant", conn)
#     display(dfsql)

,name,online_order,book_table,rate,votes,phone,primary_phone,secondary_phone,location,rest_type,dish_liked,cuisines,cost,listed_in(type),listed_in(city)
0,Jalsa,Yes,Yes,4.1,775,080 42297555\r\n+91 9743772233,080 42297555,+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,Buffet,Banashankari
1,Spice Elephant,Yes,No,4.1,787,080 41714161,080 41714161,None,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,Buffet,Banashankari
2,San Churro Cafe,Yes,No,3.8,918,+91 9663487993,+91 9663487993,None,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,Buffet,Banashankari
3,Addhuri Udupi Bhojana,No,No,3.7,88,+91 9620009302,+91 9620009302,None,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,Buffet,Banashankari
4,Grand Village,No,No,3.8,166,+91 8026612447\r\n+91 9901210005,+91 8026612447,+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,Buffet,Banashankari
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23153,Izakaya Gastro Pub,Yes,Yes,3.8,128,+91 7625087121\n+91 8050587483,+91 7625087121,+91 8050587483,Whitefield,"Bar, Casual Dining","Beer, Chicken Guntur, Paneer Tikka, Fish, Nood...","North Indian, Continental, Mediterranean",1200,Pubs and bars,Whitefield
23154,M Bar - Bengaluru Marriott Hotel Whitefield,No,No,3.9,77,080 49435000,080 49435000,None,Whitefield,"Fine Dining, Bar",Rooftop Ambience,Finger Food,2000,Pubs and bars,Whitefield
23155,Keys Cafe - Keys Hotel,No,No,2.8,161,080 39451000\n+91 8884038484,080 39451000,+91 8884038484,Whitefield,"Casual Dining, Bar","Salads, Coffee, Breakfast Buffet, Halwa, Chick...","Chinese, Continental, North Indian",1200,Pubs and bars,Whitefield
23156,Bhagini,No,No,2.5,81,080 65951222,080 65951222,None,Whitefield,"Casual Dining, Bar","Biryani, Andhra Meal","Andhra, South Indian, Chinese, North Indian",800,Pubs and bars,Whitefield
